In [1]:
import os
import sys
from pathlib import Path

In [2]:
if Path.cwd().name == "notebooks":
    os.chdir("..")

sys.path.append(Path.cwd())

Containerizing the model: Mounting Fails

In [3]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri("sqlite:///mlflow.db")
client = MlflowClient()
MODEL_NAME = "turbofan-rul"

champ = client.get_model_version_by_alias(MODEL_NAME, "champion")
run = client.get_run(champ.run_id)

print("version source", champ.source)
print("run artifacts", run.info.artifact_uri)

version source models:/m-1a723e2adcef4984ac9acaf8c067eef8
run artifacts file:///c:/dev/predictive-maintenance-mlops/mlruns/1/7ac40b18aa084306a1d54a3e73d868c0/artifacts


Exporting the model to a plain Folder

In [12]:
dest = Path("build/try/model")

local = mlflow.artifacts.download_artifacts(
    artifact_uri=f"models:/{MODEL_NAME}/{champ.version}",
    dst_path = str(dest)
)

print("Downloaded to: ", local)
print(sorted(p.name for p in Path(local).iterdir()))

Downloaded to:  c:\dev\predictive-maintenance-mlops\build\try\model
['MLmodel', 'conda.yaml', 'input_example.json', 'model.skops', 'python_env.yaml', 'registered_model_meta', 'requirements.txt', 'serving_input_example.json']


In [14]:
print((Path(local) / "requirements.txt").read_text())

mlflow==3.17.2
jaraco-text==4.0.0
numpy==2.1.3
pandas==2.2.3
psutil==7.2.2
scikit-learn==1.5.2
scipy==1.18.1
skops==0.16.0


Loading the model from the folder and predicting

In [17]:
from turbofan.data import RAW_DIR, load_cmapss
from turbofan.features import add_rolling_features, last_cycle_per_engine

model = mlflow.pyfunc.load_model(local)
expected = model.metadata.get_input_schema().input_names()
window = int(run.data.params["window"])

test_raw = load_cmapss(RAW_DIR / "test_FD001.txt")
engine = test_raw[test_raw["unit"] == 31]
latest = last_cycle_per_engine(add_rolling_features(engine, window=window))

print(round(float(model.predict(latest[expected])[0]), 1))

12.4
